# Phase 1 — 파서 & Pydantic 스키마

**목표:** Claude.ai 내보내기 JSON을 내부 `session.json`으로 변환하는 파서와 Pydantic 모델을 직접 구현한다.

**완성 후 연결:**
- `src/models.py` — 여기서 구현한 모델 클래스 붙여넣기
- `src/parser.py` — 여기서 구현한 파서 함수 붙여넣기
- Phase 2에서 서버가 이 session.json을 읽어 뷰어에 렌더링한다.

**AI 취업 포인트:**
> Pydantic은 FastAPI, LangChain, OpenAI Python SDK, Instructor 등
> 거의 모든 주요 AI 프레임워크의 핵심 의존성이다.
> 타입 안전한 데이터 파이프라인 설계 능력은 MLOps/AI 엔지니어 포지션에서 필수로 검토되는 역량이다.


---
## 섹션 1 — Pydantic이란? (이론)

### 왜 Pydantic인가?

AI 시스템에서 데이터는 항상 외부에서 들어온다 — JSON API, LLM 응답, 사용자 입력.  
타입 오류 하나가 잘못된 임베딩, 잘못된 RAG 결과로 이어진다.

```python
# 일반 딕셔너리: 런타임에 폭발
block = {"type": "thinking", "thinking": "...", "extra": None}
block["text"]  # KeyError — 어디서 터질지 모름

# Pydantic: 생성 시점에 검증
class ThinkingBlock(BaseModel):
    type: Literal["thinking"]
    text: str

ThinkingBlock(type="thinking", text="...")  # 성공
ThinkingBlock(type="text",     text="...")  # ValidationError — 즉시 감지
```

### 핵심 개념 3가지

| 개념 | 설명 |
|------|------|
| `BaseModel` | 모든 모델의 부모. `__init__` 자동 생성, 검증 내장 |
| `Literal["text"]` | 필드값을 특정 문자열로 고정 (타입 내로잉) |
| `model_dump()` | Pydantic 모델 → 딕셔너리. `model_dump_json()`은 JSON 문자열 |

### 실제 사용처

```
FastAPI      — 요청/응답 스키마, 자동 OpenAPI 문서 생성
LangChain    — Tool, Message, Runnable 등 모든 내부 타입
OpenAI SDK   — ChatCompletionMessage, ToolCall 등
Instructor   — LLM 응답을 Pydantic 모델로 파싱 (structured output)
이 프로젝트  — Block, Turn, Session 모델 → parser.py, server.py
```


---
## 섹션 2 — 데이터 구조 이해 (이론)

### conversations.json 최상위 구조

```json
[
  {
    "uuid": "d460d603-...",          // → session_id
    "name": "소프트웨어 아키텍처",    // → title
    "created_at": "2025-12-01T...",
    "updated_at": "2025-12-03T...",
    "chat_messages": [ ... ]         // → turns
  },
  ...
]
```

### chat_message 구조

```json
{
  "uuid": "019adaa2-...",
  "sender": "human",        // "human" | "assistant"  → "user" | "assistant"
  "content": [ ... ],       // 블록 목록
  "created_at": "..."
}
```

### 5가지 블록 타입

| type | 핵심 필드 | 내부 매핑 |
|------|----------|---------|
| `text` | `text` | 그대로 사용 |
| `thinking` | `thinking` | → `text`로 정규화 (주의!) |
| `tool_use` | `name`, `input` | 그대로 사용 |
| `tool_result` | `name`, `content[]`, `is_error` | content 항목을 별도 모델로 |
| `token_budget` | `remaining` | 그대로 사용, RAG 제외 대상 |

> **주의:** `thinking` 블록의 원본 필드명은 `thinking`이지만,  
> 내부 모델에서는 `text`로 정규화한다. 파서가 이 변환을 담당한다.


In [ ]:
# 데이터 탐색 (실행해서 구조 확인)
import json
from pathlib import Path

DATA_PATH = Path("data/conversations.json")

with open(DATA_PATH, encoding="utf-8") as f:
    conversations = json.load(f)

print(f"전체 대화 수: {len(conversations)}")
conv = conversations[0]
print(f"\n첫 번째 대화 키: {list(conv.keys())}")
print(f"chat_messages 수: {len(conv['chat_messages'])}")

msg = conv["chat_messages"][0]
print(f"\n첫 번째 메시지 sender: {msg['sender']}")
print(f"content 블록 수: {len(msg['content'])}")
print(f"첫 번째 블록 type: {msg['content'][0]['type']}")


In [ ]:
# 블록 타입 분포 확인
from collections import Counter

block_counter = Counter()
for conv in conversations:
    for msg in conv["chat_messages"]:
        for block in msg["content"]:
            block_counter[block["type"]] += 1

print("블록 타입 분포:")
for btype, count in block_counter.most_common():
    print(f"  {btype:15s}: {count:5d}개")


---
## 실습 1 — TextBlock, ThinkingBlock 구현

**목표:** 가장 기본적인 두 블록 타입을 Pydantic 모델로 정의한다.

```
TextBlock:     type="text",     text: str
ThinkingBlock: type="thinking", text: str  ← 원본은 "thinking" 필드지만 내부적으로 "text"로 통일
```

**힌트:**
- `Literal["text"]`로 type 필드를 고정하면 Union에서 자동 판별에 사용된다.
- `BaseModel`을 상속하면 `__init__`, 검증, `model_dump()` 모두 자동으로 생긴다.


In [ ]:
from pydantic import BaseModel
from typing import Literal

# TODO 1-1: TextBlock을 구현하세요.
# 필드: type (Literal["text"]), text (str)
class TextBlock(BaseModel):
    pass  # TODO


# TODO 1-2: ThinkingBlock을 구현하세요.
# 필드: type (Literal["thinking"]), text (str)
# 주의: 필드 이름은 text — 원본 JSON의 "thinking" 키와 다름
class ThinkingBlock(BaseModel):
    pass  # TODO


In [ ]:
# 채점 1: TextBlock
try:
    b = TextBlock(type="text", text="안녕하세요")
    assert b.type == "text", "type 필드 오류"
    assert b.text == "안녕하세요", "text 필드 오류"
    assert b.model_dump() == {"type": "text", "text": "안녕하세요"}, "직렬화 오류"
    print("✓ TextBlock 통과")
except Exception as e:
    print(f"✗ TextBlock 실패: {e}")

# 채점 2: ThinkingBlock
try:
    b = ThinkingBlock(type="thinking", text="내 생각은...")
    assert b.type == "thinking", "type 필드 오류"
    assert b.text == "내 생각은...", "text 필드 오류"
    print("✓ ThinkingBlock 통과")
except Exception as e:
    print(f"✗ ThinkingBlock 실패: {e}")

# 채점 3: 잘못된 type은 ValidationError
from pydantic import ValidationError
try:
    TextBlock(type="wrong", text="hi")
    print("✗ ValidationError가 발생해야 함")
except ValidationError:
    print("✓ Literal 타입 검증 통과")


---
## 실습 2 — ToolUseBlock, ToolResultBlock, TokenBudgetBlock, FallbackBlock

**목표:** 나머지 4가지 블록 타입을 구현한다.

**실제 데이터 예시:**
```json
// tool_use
{"type": "tool_use", "name": "artifacts", "input": {"id": "...", "content": "..."}}

// tool_result
{"type": "tool_result", "name": "artifacts", "content": [{"type": "text", "text": "OK"}], "is_error": false}

// token_budget
{"type": "token_budget", "remaining": null}
```

**힌트:**
- `dict[str, Any]`는 임의 딕셔너리를 받는 타입. `from typing import Any` 필요.
- `is_error: bool = False` 처럼 기본값 지정 가능.
- FallbackBlock은 미래에 새 블록 타입이 생겨도 데이터를 잃지 않기 위한 안전망이다.
- FallbackBlock의 `type`은 `Literal`이 아닌 `str` — 어떤 값이든 받아야 한다.


In [ ]:
from typing import Any

# TODO 2-1: ToolUseBlock
# 필드: type (Literal["tool_use"]), name (str), input (dict[str, Any])
class ToolUseBlock(BaseModel):
    pass  # TODO


# TODO 2-2: ToolResultContentItem (tool_result의 content 배열 항목)
# 필드: type (str), text (str, 기본값 "")
class ToolResultContentItem(BaseModel):
    pass  # TODO


# TODO 2-3: ToolResultBlock
# 필드: type (Literal["tool_result"]), name (str),
#        content (list[ToolResultContentItem]), is_error (bool, 기본값 False)
class ToolResultBlock(BaseModel):
    pass  # TODO


# TODO 2-4: TokenBudgetBlock
# 필드: type (Literal["token_budget"]), remaining (int | None, 기본값 None)
class TokenBudgetBlock(BaseModel):
    pass  # TODO


# TODO 2-5: FallbackBlock (알 수 없는 타입 보존용)
# 필드: type (str — Literal 아님!), raw (dict[str, Any])
class FallbackBlock(BaseModel):
    pass  # TODO


In [ ]:
# 채점: ToolUseBlock
try:
    b = ToolUseBlock(type="tool_use", name="artifacts", input={"id": "abc", "content": "..."})
    assert b.name == "artifacts"
    assert b.input["id"] == "abc"
    print("✓ ToolUseBlock 통과")
except Exception as e:
    print(f"✗ ToolUseBlock 실패: {e}")

# 채점: ToolResultBlock
try:
    item = ToolResultContentItem(type="text", text="OK")
    b = ToolResultBlock(type="tool_result", name="artifacts", content=[item], is_error=False)
    assert b.content[0].text == "OK"
    assert b.is_error == False
    # 기본값 확인
    b2 = ToolResultBlock(type="tool_result", name="artifacts", content=[])
    assert b2.is_error == False, "is_error 기본값 오류"
    print("✓ ToolResultBlock 통과")
except Exception as e:
    print(f"✗ ToolResultBlock 실패: {e}")

# 채점: TokenBudgetBlock
try:
    b1 = TokenBudgetBlock(type="token_budget", remaining=None)
    b2 = TokenBudgetBlock(type="token_budget", remaining=5000)
    assert b1.remaining is None
    assert b2.remaining == 5000
    print("✓ TokenBudgetBlock 통과")
except Exception as e:
    print(f"✗ TokenBudgetBlock 실패: {e}")

# 채점: FallbackBlock
try:
    b = FallbackBlock(type="unknown_future_type", raw={"foo": "bar", "nested": {"x": 1}})
    assert b.type == "unknown_future_type"
    assert b.raw["foo"] == "bar"
    print("✓ FallbackBlock 통과")
except Exception as e:
    print(f"✗ FallbackBlock 실패: {e}")


---
## 섹션 3 — 판별 유니온 (Discriminated Union) (이론)

블록이 5가지 타입이면, 어떻게 하나의 리스트에 담을 수 있을까?

### Union 타입

```python
from typing import Union
Block = Union[TextBlock, ThinkingBlock, ToolUseBlock, ToolResultBlock, TokenBudgetBlock, FallbackBlock]
```

`Block` 타입의 변수는 6가지 중 어느 것이든 담을 수 있다.

### Pydantic의 자동 판별

`type` 필드에 `Literal` 타입이 붙어 있으면, Pydantic은 `type` 값만 보고 올바른 모델을 선택한다.  
이를 **discriminated union** 이라 한다.

```python
# {"type": "text", "text": "hi"} → TextBlock 자동 선택
# {"type": "thinking", "thinking": "..."} → 아직 파싱 전 (파서에서 수동 처리)
```

### AI 시스템에서 이 패턴이 중요한 이유

LLM 출력은 항상 "타입이 다른 여러 조각"으로 구성된다.  
OpenAI의 `ContentBlock`, Anthropic의 `Block`, LangChain의 `Message` 모두 같은 패턴이다.

```python
# Anthropic Python SDK 실제 코드 (참고)
ContentBlock = Union[TextBlock, ToolUseBlock]
```


---
## 실습 3 — Turn, Session 모델 구현

**목표:** 블록 목록을 가진 Turn과, Turn 목록을 가진 Session을 정의한다.

```
Turn:    role ("user" | "assistant"),  blocks: list[Block]
Session: session_id, title, created_at, updated_at, turns: list[Turn]
```

**힌트:**
- `Block`은 위에서 정의한 Union 타입.
- 날짜 필드는 `str`로 저장한다 (ISO8601 문자열 그대로).


In [ ]:
from typing import Union

# Block 유니온 타입 정의
Block = Union[
    TextBlock,
    ThinkingBlock,
    ToolUseBlock,
    ToolResultBlock,
    TokenBudgetBlock,
    FallbackBlock,
]


# TODO 3-1: Turn 모델
# 필드: role (Literal["user", "assistant"]), blocks (list[Block])
class Turn(BaseModel):
    pass  # TODO


# TODO 3-2: Session 모델
# 필드: session_id (str), title (str),
#        created_at (str), updated_at (str), turns (list[Turn])
class Session(BaseModel):
    pass  # TODO


In [ ]:
# 채점: Turn
try:
    t = Turn(
        role="user",
        blocks=[TextBlock(type="text", text="안녕")]
    )
    assert t.role == "user"
    assert len(t.blocks) == 1
    assert isinstance(t.blocks[0], TextBlock)
    # 잘못된 role
    from pydantic import ValidationError
    try:
        Turn(role="bot", blocks=[])
        print("✗ role 검증 실패")
    except ValidationError:
        pass
    print("✓ Turn 통과")
except Exception as e:
    print(f"✗ Turn 실패: {e}")

# 채점: Session
try:
    s = Session(
        session_id="abc-123",
        title="테스트 세션",
        created_at="2025-12-01T00:00:00Z",
        updated_at="2025-12-01T01:00:00Z",
        turns=[
            Turn(role="user",      blocks=[TextBlock(type="text", text="질문")]),
            Turn(role="assistant", blocks=[TextBlock(type="text", text="답변")]),
        ]
    )
    assert s.session_id == "abc-123"
    assert len(s.turns) == 2
    assert s.turns[0].role == "user"

    # JSON 직렬화 확인
    j = s.model_dump_json(indent=2)
    parsed = json.loads(j)
    assert parsed["session_id"] == "abc-123"
    print("✓ Session 통과")
    print("\nJSON 직렬화 미리보기 (처음 300자):")
    print(j[:300] + "...")
except Exception as e:
    print(f"✗ Session 실패: {e}")


---
## 섹션 4 — 파서 설계 (이론)

### 변환 흐름

```
conversations.json
  └── conversation[]
        ├── uuid       → session_id
        ├── name       → title
        ├── created_at
        ├── updated_at
        └── chat_messages[]
              ├── sender  → role  (human→user, assistant→assistant)
              └── content[]
                    └── block  → parse_block() → Block 모델
```

### parse_block() 설계

```python
def parse_block(raw: dict) -> Block:
    t = raw.get("type")
    if t == "text":
        return TextBlock(type="text", text=raw["text"])
    elif t == "thinking":
        return ThinkingBlock(type="thinking", text=raw["thinking"])  # ← 필드명 변환!
    elif ...
```

### 왜 if/elif 로 수동 분기하는가?

`thinking` 블록은 원본 필드명이 `thinking`이지만 내부 모델은 `text`를 사용한다.  
Pydantic의 자동 판별 대신 수동 변환이 필요한 케이스다.

→ 이런 변환 로직은 `parser.py`에만 있고, 나머지 코드는 항상 정규화된 모델만 다룬다.  
이것이 파서 계층을 두는 이유다.


---
## 실습 4 — parse_block() 함수 구현

**목표:** raw 딕셔너리를 읽어 올바른 Block 모델 인스턴스를 반환한다.

**힌트:**
- `type` 필드로 분기.
- `thinking` 블록: `raw["thinking"]` → `ThinkingBlock(text=...)`
- 알 수 없는 타입: `FallbackBlock(type=t, raw=raw)` 반환


In [ ]:
def parse_block(raw: dict) -> Block:
    """
    raw 블록 딕셔너리를 Block 모델로 변환한다.
    알 수 없는 type은 FallbackBlock으로 보존한다.
    """
    block_type = raw.get("type")

    if block_type == "text":
        # TODO: TextBlock 반환
        pass  # TODO

    elif block_type == "thinking":
        # TODO: ThinkingBlock 반환
        # 힌트: raw["thinking"] → text 필드로
        pass  # TODO

    elif block_type == "tool_use":
        # TODO: ToolUseBlock 반환
        pass  # TODO

    elif block_type == "tool_result":
        # TODO: ToolResultBlock 반환
        # 힌트: raw["content"]의 각 항목을 ToolResultContentItem으로 변환
        pass  # TODO

    elif block_type == "token_budget":
        # TODO: TokenBudgetBlock 반환
        pass  # TODO

    else:
        # TODO: FallbackBlock 반환
        pass  # TODO


In [ ]:
# 채점: parse_block

# text 블록
raw_text = {"type": "text", "text": "안녕하세요", "start_timestamp": "...", "citations": []}
b = parse_block(raw_text)
assert isinstance(b, TextBlock), f"TextBlock이어야 함, 실제: {type(b)}"
assert b.text == "안녕하세요"
print("✓ text 블록 통과")

# thinking 블록 (필드명 변환 핵심!)
raw_thinking = {"type": "thinking", "thinking": "내 생각은...", "summaries": []}
b = parse_block(raw_thinking)
assert isinstance(b, ThinkingBlock), f"ThinkingBlock이어야 함, 실제: {type(b)}"
assert b.text == "내 생각은...", f"thinking→text 변환 실패, b.text={b.text!r}"
print("✓ thinking 블록 통과 (필드 변환 확인)")

# tool_use 블록
raw_tool = {"type": "tool_use", "name": "artifacts", "input": {"content": "print('hi')"}}
b = parse_block(raw_tool)
assert isinstance(b, ToolUseBlock)
assert b.name == "artifacts"
print("✓ tool_use 블록 통과")

# tool_result 블록
raw_result = {
    "type": "tool_result", "name": "artifacts",
    "content": [{"type": "text", "text": "OK"}],
    "is_error": False
}
b = parse_block(raw_result)
assert isinstance(b, ToolResultBlock)
assert b.content[0].text == "OK"
print("✓ tool_result 블록 통과")

# token_budget 블록
raw_budget = {"type": "token_budget", "remaining": None}
b = parse_block(raw_budget)
assert isinstance(b, TokenBudgetBlock)
assert b.remaining is None
print("✓ token_budget 블록 통과")

# 알 수 없는 타입 → FallbackBlock
raw_unknown = {"type": "future_block", "foo": "bar"}
b = parse_block(raw_unknown)
assert isinstance(b, FallbackBlock), f"FallbackBlock이어야 함, 실제: {type(b)}"
assert b.type == "future_block"
print("✓ FallbackBlock 통과")

print("\n✓✓✓ parse_block() 전체 통과!")


---
## 실습 5 — parse_conversation() 함수 구현

**목표:** 단일 conversation 딕셔너리를 Session 모델로 완성 변환한다.

**변환 규칙:**
```
conv["uuid"]          → session.session_id
conv["name"]          → session.title
conv["created_at"]    → session.created_at
conv["updated_at"]    → session.updated_at
conv["chat_messages"] → session.turns  (각 메시지를 Turn으로)

msg["sender"] 변환:
  "human"     → "user"
  "assistant" → "assistant"
```

**힌트:**
- `SENDER_TO_ROLE = {"human": "user", "assistant": "assistant"}`
- 각 메시지의 `content` 배열을 `parse_block()`으로 변환


In [ ]:
SENDER_TO_ROLE = {"human": "user", "assistant": "assistant"}


def parse_message(msg: dict) -> Turn:
    """chat_message 딕셔너리 → Turn 모델"""
    # TODO: sender → role 변환, content 블록 목록 변환
    pass  # TODO


def parse_conversation(conv: dict) -> Session:
    """conversation 딕셔너리 → Session 모델"""
    # TODO: uuid→session_id, name→title, chat_messages→turns
    pass  # TODO


In [ ]:
# 채점: parse_conversation (실제 데이터로 검증)

session = parse_conversation(conversations[0])

assert isinstance(session, Session), f"Session이어야 함, 실제: {type(session)}"
assert session.session_id == conversations[0]["uuid"], "session_id 불일치"
assert session.title == conversations[0]["name"], "title 불일치"
assert len(session.turns) == len(conversations[0]["chat_messages"]), "turns 수 불일치"

# role 변환 확인
first_turn = session.turns[0]
first_msg  = conversations[0]["chat_messages"][0]
expected_role = SENDER_TO_ROLE[first_msg["sender"]]
assert first_turn.role == expected_role, f"role 불일치: {first_turn.role!r} != {expected_role!r}"

# 블록 수 확인
first_blocks = first_msg["content"]
assert len(first_turn.blocks) == len(first_blocks), "블록 수 불일치"

print(f"✓ session_id: {session.session_id}")
print(f"✓ title: {session.title}")
print(f"✓ turns: {len(session.turns)}개")
print(f"✓ 첫 번째 turn role: {session.turns[0].role}")
print(f"✓ 첫 번째 turn 블록 수: {len(session.turns[0].blocks)}")

# 블록 타입 분포 출력
block_dist = Counter(type(b).__name__ for t in session.turns for b in t.blocks)
print(f"\n블록 타입 분포:")
for btype, cnt in block_dist.most_common():
    print(f"  {btype}: {cnt}")


---
## 최종 통합 — 전체 데이터셋 파싱

224개 대화 전체를 파싱해 오류 없이 통과하면 Phase 1 완료.


In [ ]:
# 전체 파싱 + 통계
errors = []
block_type_dist = Counter()

for i, conv in enumerate(conversations):
    try:
        session = parse_conversation(conv)
        for turn in session.turns:
            for block in turn.blocks:
                block_type_dist[type(block).__name__] += 1
    except Exception as e:
        errors.append((i, conv.get("uuid"), str(e)))

print(f"전체 대화: {len(conversations)}개")
print(f"성공: {len(conversations) - len(errors)}개")
print(f"실패: {len(errors)}개")

if errors:
    print("\n실패 목록:")
    for idx, uid, err in errors[:5]:
        print(f"  [{idx}] {uid}: {err}")
else:
    print("\n✓ 모든 대화 파싱 성공!")

print("\n블록 타입 분포:")
for btype, cnt in block_type_dist.most_common():
    print(f"  {btype:30s}: {cnt:5d}개")


---
## 연결 — 다음 단계

### 이 노트북에서 구현한 것 → .py 파일로 옮기기

| 구현한 것 | 옮길 파일 |
|----------|-----------|
| 블록 모델 클래스들, Turn, Session | `src/models.py` |
| `parse_block()`, `parse_message()`, `parse_conversation()` | `src/parser.py` |

### Phase 2에서 어디에 쓰이는가

```
server.py
  GET /api/sessions
    → conversations/ 디렉토리의 session.json 목록 반환

parser.py  ← 이번에 구현
  parse_export()
    → conversations.json → session.json 파일들 저장

뷰어 (app.js)
  → GET /api/session/<id>
  → session.json의 turns[].blocks[]를 렌더링
```

### AI 취업 연결

이 파서 구조는 실무에서 매우 흔하다:

```
OpenAI API 응답 파싱     → ChatCompletion.choices[].message.content[]
Anthropic API 응답 파싱  → Message.content[] (TextBlock | ToolUseBlock)
LangChain Document 파싱  → Document(page_content, metadata)
```

모두 외부 JSON → 내부 타입 안전 모델 패턴이다.  
Pydantic + discriminated union + 변환 계층이 표준 해법이다.
